# ATML PA2 - GPU runner (Colab / Kaggle)

Runs every experiment with `scripts/run_all.sh`. Re-running a cell after a disconnect resumes: finished
adapters/results are skipped. On Colab, `outputs/` and `results/` live on Google Drive so they survive
session resets. Use a GPU runtime (T4 is enough; A100/L4 is ~3x faster).

In [ ]:
REPO_URL = "https://github.com/laeebahafeez/ATML-PA2.git"   # <-- your fork/repo
BRANCH = "main"
ON_COLAB = True          # False on Kaggle (then results persist via 'Save Version' of /kaggle/working)

In [ ]:
import os, subprocess
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST = '/content/drive/MyDrive/atml_pa2_runs'
    WORK = '/content/pa2'
else:
    PERSIST = '/kaggle/working/atml_pa2_runs'
    WORK = '/kaggle/working/pa2'
os.makedirs(PERSIST, exist_ok=True)
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, WORK], check=True)
%cd {WORK}
!git pull --ff-only
# keep trained adapters + result files on persistent storage
for d in ['outputs', 'results']:
    os.makedirs(f'{PERSIST}/{d}', exist_ok=True)
    if os.path.isdir(d) and not os.path.islink(d):
        !cp -rn {d}/. {PERSIST}/{d}/ && rm -rf {d}
    if not os.path.exists(d):
        os.symlink(f'{PERSIST}/{d}', d)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install -r requirements.txt
!python -m scripts.download_assets
!python -m scripts.validate_assets
!python -m scripts.check_environment
!python -m pytest tests -q

Run stages one at a time (each is resumable). Approximate T4 times: t1 ~1.7h, t2 ~1.1h, t3 ~0.8h, t4 ~1h, t5 ~1.2h.

In [ ]:
!bash scripts/run_all.sh t1 2>&1 | tee -a results/run_t1.log

In [ ]:
!bash scripts/run_all.sh t2 2>&1 | tee -a results/run_t2.log

In [ ]:
!bash scripts/run_all.sh t3 2>&1 | tee -a results/run_t3.log

In [ ]:
!bash scripts/run_all.sh t4 2>&1 | tee -a results/run_t4.log

In [ ]:
!bash scripts/run_all.sh t5 2>&1 | tee -a results/run_t5.log

## Manual audit (Task 4)
Download `results/task4_safety/manual_audit_sheet.csv`, fill `manual_label` (see `MANUAL_AUDIT_GUIDE.txt`) **without
looking at the judged files**, upload it back to the same path, then run the cell below.

In [ ]:
!python -m task4_safety.evaluate_safety
!python -m report.make_figures

In [ ]:
# bundle small result files (no weights) to download / commit
!tar czf {PERSIST}/results_bundle.tgz results report/figures report/tables.md
print('bundle at', PERSIST + '/results_bundle.tgz')